# 📘 Módulo 16 — RAG, Memória e Sistemas de IA Híbridos

Este módulo mostra como construir sistemas de IA modernos que combinam:

- **LLMs (modelos de linguagem)**
- **Bases de conhecimento**
- **Vetores e embeddings**
- **Memória de longo prazo**
- **RAG (Retrieval-Augmented Generation)**

Você aprenderá a criar sistemas como:
- Chatbots corporativos com base documental
- Assistentes que lembram informações
- Sistemas de busca semântica
- Aplicações que combinam IA + banco de dados

---

# 🎯 Objetivos do Módulo 16

Ao final deste módulo, você será capaz de:

✔️ entender o que é RAG e por que ele é essencial  
✔️ criar embeddings e armazená-los em um vetor store  
✔️ implementar busca semântica  
✔️ integrar RAG com modelos como GPT, T5 e LLaMA  
✔️ criar memória de longo prazo para um assistente  
✔️ construir um sistema híbrido completo  

---

# 📚 Índice

1. [O que é RAG? Por que ele é essencial?](#rag)
2. [Embeddings e Espaços Vetoriais](#embeddings)
3. [Vector Stores (FAISS)](#faiss)
4. [Busca Semântica na Prática](#busca)
5. [Construindo um Pipeline RAG Completo](#pipeline)
6. [Memória de Longo Prazo para Assistentes](#memoria)
7. [Arquiteturas Híbridas (LLM + Banco de Dados)](#hibrido)
8. [Projeto Final — Chatbot com RAG + Memória](#projeto)

---

# 0. Setup — bibliotecas

Vamos usar:
- HuggingFace Transformers
- SentenceTransformers
- FAISS (Facebook AI Similarity Search)

(Tudo leve e compatível com CPU.)


In [ ]:
import numpy as np
import faiss
import tensorflow as tf
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt

tf.__version__

<a id="embeddings"></a>
# 2. Embeddings e Espaços Vetoriais

Embeddings são a base de todo sistema moderno de IA que envolve:
- RAG
- busca semântica
- recomendação
- clustering
- memória de longo prazo

Eles transformam **texto** em **vetores numéricos** que capturam significado.

Exemplo:
- "cachorro" e "gato" ficam próximos
- "carro" e "automóvel" ficam próximos
- "amor" e "ódio" ficam distantes

Vamos entender isso na prática.

---
# 2.1 O que é um embedding?

É um vetor de alta dimensão, por exemplo:

```
[0.12, -0.44, 0.88, ..., 0.03]
```

Cada vetor representa o **significado** de um texto.

Modelos famosos de embeddings:
- Sentence-BERT
- MiniLM
- E5
- GTE

Vamos usar **sentence-transformers** (leve e rápido).

---
# 2.2 Instalando SentenceTransformers

In [ ]:
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer

---
# 2.3 Carregando um modelo de embeddings

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

---
# 2.4 Criando embeddings de frases

In [ ]:
sentences = [
    "O gato está no telhado.",
    "O cachorro está no quintal.",
    "A economia global está crescendo.",
    "O mercado financeiro caiu hoje."
]

emb = model.encode(sentences)
emb.shape

🟣 **Interpretação**

Cada frase virou um vetor de dimensão 384.

Agora podemos medir similaridade entre elas.

---
# 2.5 Similaridade de embeddings (cosine similarity)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity([emb[0]], emb)[0]
sim

🟣 **Interpretação**

O vetor 0 ("gato no telhado") é mais parecido com:
- "cachorro no quintal"  
do que com:
- "economia global"

Isso mostra que embeddings capturam **significado**, não palavras exatas.

---
# 2.6 Visualização da similaridade

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(6,4))
plt.bar(range(len(sim)), sim)
plt.xticks(range(len(sim)), ["gato", "cachorro", "economia", "mercado"])
plt.title("Similaridade com 'O gato está no telhado'")
plt.show()

---
# 2.7 Por que embeddings são essenciais para RAG?

Porque RAG precisa:
- encontrar documentos relevantes
- mesmo que não contenham as mesmas palavras

Exemplo:
- pergunta: "Como treinar um modelo de IA?"
- documento: "Processo de machine learning"

Mesmo sem palavras iguais, embeddings capturam a relação.

---
# 2.8 Embeddings permitem:

✔️ busca semântica  
✔️ recomendação inteligente  
✔️ clustering de documentos  
✔️ detecção de duplicatas  
✔️ memória de longo prazo  
✔️ RAG (Retrieval-Augmented Generation)  

Eles são o **motor semântico** dos sistemas modernos.

---
# 2.9 Conclusão desta parte

✔️ Entendemos o que são embeddings  
✔️ Geramos embeddings com SentenceTransformers  
✔️ Calculamos similaridade semântica  
✔️ Visualizamos relações entre frases  
✔️ Entendemos por que embeddings são essenciais para RAG  

Agora estamos prontos para:

**PARTE 3 — Vector Stores (FAISS): armazenando e buscando vetores.**

<a id="faiss"></a>
# 3. Vector Stores (FAISS)

Agora que você já sabe gerar embeddings, precisamos de um lugar para:

- armazenar vetores
- indexá-los
- buscar os mais semelhantes

Esse lugar é chamado de:

# 👉 Vector Store

O mais famoso e eficiente é o **FAISS**, criado pelo Facebook AI Research.

---
# 3.1 Instalando FAISS

In [ ]:
!pip install faiss-cpu --quiet

import faiss
import numpy as np

---
# 3.2 Criando embeddings de exemplo

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

docs = [
    "O gato está no telhado.",
    "O cachorro está no quintal.",
    "A economia global está crescendo.",
    "O mercado financeiro caiu hoje.",
    "Gatos gostam de dormir no sol.",
]

emb = model.encode(docs)
emb.shape

---
# 3.3 Criando um índice FAISS

O índice mais simples é o **IndexFlatL2**, que usa distância euclidiana.

In [ ]:
dim = emb.shape[1]  # dimensão dos vetores

index = faiss.IndexFlatL2(dim)
index.add(emb)

index.ntotal  # número de vetores armazenados

---
# 3.4 Buscando vetores semelhantes

Vamos buscar os documentos mais parecidos com:

```
"gato no sol"
```

In [ ]:
query = model.encode(["gato no sol"])

distances, indices = index.search(query, k=3)
distances, indices

---
# 3.5 Interpretando resultados

for i in indices[0]:
    print(f"- {docs[i]}")

🟣 **Interpretação**

Mesmo sem palavras idênticas, FAISS encontrou:

- frases sobre **gatos**
- frases sobre **sol** ou **comportamento de gatos**

Isso é **busca semântica real**.

---
# 3.6 Visualização da busca

import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.bar(range(len(distances[0])), distances[0])
plt.xticks(range(len(distances[0])), indices[0])
plt.title("Distâncias dos documentos mais próximos")
plt.show()

---
# 3.7 Por que FAISS é tão importante?

✔️ extremamente rápido  
✔️ suporta milhões de vetores  
✔️ permite busca aproximada (ANN)  
✔️ funciona em CPU e GPU  
✔️ é padrão da indústria  

Sem FAISS, RAG não escala para bases grandes.

---
# 3.8 Tipos de índices FAISS (visão geral)

| Índice | Descrição | Uso |
|--------|-----------|------|
| `IndexFlatL2` | exato, simples | bases pequenas |
| `IVF` | inverted file index | bases médias |
| `HNSW` | grafo de vizinhos | bases grandes |
| `PQ` | product quantization | compressão |
| `GPUIndex` | versão GPU | máxima performance |

Para este curso, usaremos `IndexFlatL2`, mas você já sabe que existem opções mais avançadas.

---
# 3.9 Conclusão desta parte

✔️ Entendemos o que é um Vector Store  
✔️ Instalamos e usamos FAISS  
✔️ Criamos um índice vetorial  
✔️ Fizemos busca semântica real  
✔️ Vimos por que FAISS é essencial para RAG  

Agora estamos prontos para:

**PARTE 4 — Busca Semântica na Prática (RAG passo 1).**

<a id="busca"></a>
# 4. Busca Semântica na Prática

Agora vamos juntar:
- embeddings (significado)
- FAISS (busca rápida)

Para criar um sistema de **busca semântica**.

Objetivo:
> Dado um texto de consulta, encontrar os documentos mais relevantes
  com base no significado, não nas palavras exatas.

---
# 4.1 Criando uma base de documentos

docs = [
    "Machine learning é uma área da inteligência artificial.",
    "Redes neurais são modelos inspirados no cérebro humano.",
    "A economia global está passando por mudanças importantes.",
    "O mercado financeiro reagiu à nova política monetária.",
    "Transformers são modelos avançados de deep learning.",
    "O gato dormiu no sofá durante a tarde.",
    "Cachorros são animais muito leais."
]

docs

---
# 4.2 Gerando embeddings dos documentos

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = model.encode(docs)

doc_embeddings.shape

---
# 4.3 Criando um índice FAISS

import faiss

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(doc_embeddings)

index.ntotal

---
# 4.4 Função de busca semântica

def search(query, k=3):
    q_emb = model.encode([query])
    distances, indices = index.search(q_emb, k)
    results = [(docs[i], float(distances[0][j])) for j, i in enumerate(indices[0])]
    return results

---
# 4.5 Testando a busca semântica

search("O que são redes neurais?", k=3)

---
# 4.6 Outro teste

search("Como funciona deep learning?", k=3)

---
# 4.7 Teste com tema totalmente diferente

search("Animais domésticos", k=3)

---
# 4.8 Visualização dos resultados

query = "Como funciona deep learning?"
results = search(query, k=4)

import matplotlib.pyplot as plt

labels = [r[0][:25] + "..." for r in results]
scores = [r[1] for r in results]

plt.figure(figsize=(8,4))
plt.barh(labels, scores)
plt.gca().invert_yaxis()
plt.title(f"Similaridade com a consulta:\n'{query}'")
plt.show()

---
# 4.9 O que aprendemos aqui?

✔️ Criamos uma base de documentos  
✔️ Geramos embeddings  
✔️ Indexamos tudo com FAISS  
✔️ Implementamos busca semântica real  
✔️ Vimos que o sistema entende **significado**, não palavras  

Isso é exatamente o que motores modernos como:
- ChatGPT RAG  
- Perplexity  
- Bing Copilot  
- Assistentes corporativos  

fazem nos bastidores.

---
# 4.10 Conclusão desta parte

✔️ Você agora tem um mecanismo de busca semântica funcional  
✔️ Ele já pode ser integrado a um LLM  
✔️ Este é o **primeiro passo do RAG completo**  

Agora estamos prontos para:

**PARTE 5 — Construindo um Pipeline RAG Completo (Busca + LLM).**

<a id="pipeline"></a>
# 5. Construindo um Pipeline RAG Completo

Agora vamos unir tudo:

1. Embeddings → significado  
2. FAISS → busca semântica  
3. LLM → geração de resposta  

O fluxo completo é:

```
Pergunta → Embedding → Busca FAISS → Documentos relevantes → LLM → Resposta final
```

---
# 5.1 Base de conhecimento (documentos)

docs = [
    "Machine learning é uma área da IA que cria modelos capazes de aprender padrões.",
    "Redes neurais são compostas por camadas de neurônios artificiais.",
    "Transformers utilizam mecanismos de atenção para processar sequências.",
    "O mercado financeiro reage a mudanças na política monetária.",
    "Deep learning é um subconjunto do machine learning baseado em redes neurais profundas.",
    "A economia global está passando por transformações tecnológicas.",
]

docs

---
# 5.2 Gerando embeddings dos documentos

from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(docs)

doc_embeddings.shape

---
# 5.3 Criando índice FAISS

import faiss

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(doc_embeddings)

index.ntotal

---
# 5.4 Função de busca semântica

def retrieve(query, k=3):
    q_emb = embedder.encode([query])
    distances, indices = index.search(q_emb, k)
    return [docs[i] for i in indices[0]]

---
# 5.5 Carregando um LLM para gerar respostas

Usaremos **FLAN‑T5**, leve e ótimo para instruções.

In [ ]:
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 5.6 Função RAG completa

Pipeline:

1. Busca documentos relevantes  
2. Monta um prompt com contexto  
3. LLM gera a resposta  

In [ ]:
def rag_answer(query, k=3):
    retrieved_docs = retrieve(query, k)
    context = "\n".join(retrieved_docs)

    prompt = f"""
Use as informações abaixo para responder a pergunta.

Contexto:
{context}

Pergunta:
{query}

Resposta:
"""

    tokens = tokenizer(prompt, return_tensors="tf", truncation=True)
    output = llm.generate(**tokens, max_length=150)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 5.7 Testando o RAG

rag_answer("O que são redes neurais?")

---
# 5.8 Outro teste

rag_answer("Explique o que é deep learning.")

---
# 5.9 Teste com pergunta sobre economia

rag_answer("O que está acontecendo com a economia global?")

---
# 5.10 Visualização do fluxo RAG

import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.text(0.1, 0.5, "Pergunta\n(usuário)", fontsize=14)
plt.text(3.1, 0.5, "Embeddings\n+ FAISS", fontsize=14)
plt.text(6.1, 0.5, "Documentos\nrelevantes", fontsize=14)
plt.text(8.8, 0.5, "LLM\n(resposta)", fontsize=14)
plt.axis("off")
plt.title("Fluxo completo do RAG")
plt.show()

---
# 5.11 O que você acabou de construir?

✔️ Um sistema de **busca semântica**  
✔️ Um **vector store** com FAISS  
✔️ Um pipeline **RAG completo**  
✔️ Um LLM que responde usando **contexto real**  

Isso é exatamente o que empresas usam para:
- chatbots corporativos  
- atendimento ao cliente  
- busca inteligente em PDFs  
- sistemas de suporte técnico  
- assistentes internos com base documental  

---
# 5.12 Conclusão desta parte

✔️ Você construiu um RAG funcional  
✔️ Ele já responde perguntas com base em documentos  
✔️ Este é o núcleo de qualquer chatbot corporativo  

Agora estamos prontos para:

**PARTE 6 — Memória de Longo Prazo para Assistentes.**

<a id="memoria"></a>
# 6. Memória de Longo Prazo para Assistentes

Até agora, você construiu:
- embeddings
- FAISS
- busca semântica
- pipeline RAG completo

Agora vamos adicionar **memória**, que é diferente de RAG:

- RAG → busca em documentos externos  
- Memória → informações sobre o usuário e o histórico  

Um assistente com memória pode:
- lembrar preferências
- lembrar fatos importantes
- acompanhar tarefas
- personalizar respostas

---
# 6.1 Como funciona a memória em assistentes modernos?

A memória é armazenada como **vetores**, exatamente como documentos.

Cada memória é um texto curto:

- "O usuário gosta de gatos."
- "O usuário trabalha com marketing."
- "O usuário está estudando IA."

E cada memória vira um embedding.

Depois, usamos FAISS para recuperar memórias relevantes.

---
# 6.2 Criando uma memória inicial

memories = [
    "O usuário gosta de gatos.",
    "O usuário trabalha com análise de dados.",
    "O usuário está aprendendo machine learning.",
    "O usuário prefere respostas curtas e diretas.",
]

memories

---
# 6.3 Gerando embeddings das memórias

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embedder = SentenceTransformer("all-MiniLM-L6-v2")
mem_emb = embedder.encode(memories)

mem_emb.shape

---
# 6.4 Criando índice FAISS para memória

dim = mem_emb.shape[1]
memory_index = faiss.IndexFlatL2(dim)
memory_index.add(mem_emb)

memory_index.ntotal

---
# 6.5 Função para recuperar memórias relevantes

def recall_memory(query, k=2):
    q_emb = embedder.encode([query])
    distances, indices = memory_index.search(q_emb, k)
    return [memories[i] for i in indices[0]]

---
# 6.6 Testando a memória

recall_memory("Quero aprender mais sobre IA.")

---
# 6.7 Outro teste

recall_memory("Gatos são legais.")

---
# 6.8 Adicionando nova memória dinamicamente

Assistentes reais **aprendem ao longo do tempo**.

In [ ]:
def add_memory(text):
    global memories, mem_emb, memory_index

    memories.append(text)
    new_emb = embedder.encode([text])

    # adiciona ao índice FAISS
    memory_index.add(new_emb)

    return "Memória adicionada."

add_memory("O usuário gosta de estudar à noite.")

---
# 6.9 Testando após adicionar memória

recall_memory("Qual é meu horário preferido de estudo?")

---
# 6.10 Integrando memória ao RAG

Agora vamos combinar:
- documentos (RAG)
- memórias pessoais

O assistente usa **ambos** para responder.

In [ ]:
def rag_with_memory(query, k_docs=2, k_mem=2):
    # Recupera documentos
    retrieved_docs = retrieve(query, k_docs)

    # Recupera memórias
    retrieved_mem = recall_memory(query, k_mem)

    context = "\n".join(retrieved_docs + retrieved_mem)

    prompt = f"""
Use as informações abaixo para responder a pergunta.

Contexto:
{context}

Pergunta:
{query}

Resposta:
"""

    tokens = tokenizer(prompt, return_tensors="tf", truncation=True)
    output = llm.generate(**tokens, max_length=150)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 6.11 Testando o assistente com memória

rag_with_memory("O que você sabe sobre meus estudos?")

---
# 6.12 Outro teste

rag_with_memory("Qual animal eu gosto?")

---
# 6.13 O que você acabou de construir?

✔️ Um sistema de memória vetorial  
✔️ Capaz de armazenar preferências do usuário  
✔️ Capaz de recuperar memórias relevantes  
✔️ Capaz de aprender novas memórias  
✔️ Integrado ao pipeline RAG  

Isso é exatamente o que assistentes modernos fazem:
- Chatbots corporativos  
- Assistentes pessoais  
- Sistemas de recomendação  
- Agentes autônomos  

---
# 6.14 Conclusão desta parte

✔️ Você criou memória de longo prazo  
✔️ Aprendeu a armazenar e recuperar memórias  
✔️ Aprendeu a atualizar memórias dinamicamente  
✔️ Integrar memória + RAG + LLM  

Agora estamos prontos para:

**PARTE 7 — Arquiteturas Híbridas (LLM + Banco de Dados + RAG + Memória).**

<a id="hibrido"></a>
# 7. Arquiteturas Híbridas (LLM + Banco de Dados + RAG + Memória)

Até agora você construiu:
- embeddings
- FAISS
- RAG completo
- memória de longo prazo

Agora vamos integrar tudo isso com:
- **bancos de dados tradicionais**
- **regras de negócio**
- **funções externas**

Esse é o padrão moderno de sistemas de IA:

```
Usuário → LLM → (RAG + Memória + Banco de Dados) → Resposta
```

---
# 7.1 Por que arquiteturas híbridas são necessárias?

Porque LLMs **não são bancos de dados**.

Eles:
- não garantem precisão factual
- não armazenam dados estruturados
- não fazem cálculos confiáveis
- não têm memória persistente

Por isso, precisamos combinar:

- **LLM** → linguagem, raciocínio, geração  
- **RAG** → conhecimento externo  
- **Memória** → preferências do usuário  
- **Banco de dados** → dados estruturados  
- **Regras** → lógica de negócio  

---
# 7.2 Criando um banco de dados simples (SQLite)

import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE clientes (
    id INTEGER PRIMARY KEY,
    nome TEXT,
    cidade TEXT,
    saldo REAL
)
""")

cursor.executemany("""
INSERT INTO clientes (nome, cidade, saldo)
VALUES (?, ?, ?)
""", [
    ("Ana", "São Paulo", 1500.0),
    ("Bruno", "Campinas", 2300.0),
    ("Carla", "Indaiatuba", 900.0),
])

conn.commit()

---
# 7.3 Função para consultar o banco de dados

def consultar_cliente(nome):
    cursor.execute("SELECT * FROM clientes WHERE nome = ?", (nome,))
    row = cursor.fetchone()
    if row:
        return f"Cliente: {row[1]}, Cidade: {row[2]}, Saldo: R$ {row[3]:.2f}"
    return "Cliente não encontrado."

consultar_cliente("Carla")

---
# 7.4 Integrando LLM + Banco de Dados

O LLM decide **quando** consultar o banco.

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

def llm_decide(query):
    prompt = f"""
Você é um assistente inteligente.
Se a pergunta for sobre clientes, responda com: "CONSULTAR: <nome>"
Caso contrário, responda normalmente.

Pergunta: {query}
Resposta:
"""
    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=50)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 7.5 Pipeline híbrido completo

def hybrid_assistant(query):
    decision = llm_decide(query)

    if decision.startswith("CONSULTAR:"):
        nome = decision.replace("CONSULTAR:", "").strip()
        return consultar_cliente(nome)

    # Caso não seja consulta ao banco, usa RAG + memória
    return rag_with_memory(query)

---
# 7.6 Testando o assistente híbrido

hybrid_assistant("Qual é o saldo da Carla?")

---
# 7.7 Outro teste (não envolve banco)

hybrid_assistant("O que você sabe sobre meus estudos?")

---
# 7.8 Outro teste (mistura tudo)

hybrid_assistant("Explique deep learning e depois diga meu animal favorito.")

---
# 7.9 Visualização da arquitetura híbrida

import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.text(0.1, 0.7, "Usuário", fontsize=14)
plt.text(3.1, 0.7, "LLM\n(decisão)", fontsize=14)
plt.text(6.1, 0.9, "RAG\n(documentos)", fontsize=14)
plt.text(6.1, 0.5, "Memória\n(preferências)", fontsize=14)
plt.text(6.1, 0.1, "Banco de Dados\n(dados estruturados)", fontsize=14)
plt.text(8.8, 0.7, "Resposta Final", fontsize=14)
plt.axis("off")
plt.title("Arquitetura Híbrida Completa")
plt.show()

---
# 7.10 O que você acabou de construir?

✔️ Um assistente que usa **RAG**  
✔️ Um assistente com **memória de longo prazo**  
✔️ Um assistente que consulta **banco de dados real**  
✔️ Um sistema que decide **qual fonte usar**  
✔️ Uma arquitetura híbrida moderna  

Isso é exatamente o que empresas usam hoje para:
- atendimento ao cliente  
- suporte técnico  
- assistentes internos  
- agentes autônomos  
- sistemas de análise inteligente  

---
# 7.11 Conclusão desta parte

✔️ Você aprendeu a integrar LLM + RAG + Memória + Banco de Dados  
✔️ Criou um assistente híbrido funcional  
✔️ Entendeu a arquitetura usada em sistemas reais  

Agora estamos prontos para:

**PARTE 8 — Projeto Final: Chatbot com RAG + Memória + Banco de Dados.**

# 8. Projeto Final — Chatbot com RAG + Memória + Banco de Dados

Este projeto integra:

✔️ RAG (documentos externos)  
✔️ Memória de longo prazo (preferências do usuário)  
✔️ Banco de dados (informações estruturadas)  
✔️ LLM (decisão + geração de resposta)  

Você está construindo um **assistente completo**, como os usados em empresas.

---
# 8.1 Setup — importações

import sqlite3
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

---
# 8.2 Criando o banco de dados (clientes)

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE clientes (
    id INTEGER PRIMARY KEY,
    nome TEXT,
    cidade TEXT,
    saldo REAL
)
""")

cursor.executemany("""
INSERT INTO clientes (nome, cidade, saldo)
VALUES (?, ?, ?)
""", [
    ("Ana", "São Paulo", 1500.0),
    ("Bruno", "Campinas", 2300.0),
    ("Carla", "Indaiatuba", 900.0),
])

conn.commit()

def consultar_cliente(nome):
    cursor.execute("SELECT * FROM clientes WHERE nome = ?", (nome,))
    row = cursor.fetchone()
    if row:
        return f"Cliente: {row[1]}, Cidade: {row[2]}, Saldo: R$ {row[3]:.2f}"
    return "Cliente não encontrado."

---
# 8.3 Criando a base documental (RAG)

docs = [
    "Machine learning é uma área da IA que cria modelos capazes de aprender padrões.",
    "Redes neurais são compostas por camadas de neurônios artificiais.",
    "Transformers utilizam mecanismos de atenção para processar sequências.",
    "Deep learning é um subconjunto do machine learning baseado em redes neurais profundas.",
    "A economia global está passando por transformações tecnológicas.",
]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_emb = embedder.encode(docs)

dim = doc_emb.shape[1]
rag_index = faiss.IndexFlatL2(dim)
rag_index.add(doc_emb)

def retrieve_docs(query, k=3):
    q_emb = embedder.encode([query])
    distances, indices = rag_index.search(q_emb, k)
    return [docs[i] for i in indices[0]]

---
# 8.4 Criando memória de longo prazo

memories = [
    "O usuário gosta de gatos.",
    "O usuário trabalha com análise de dados.",
    "O usuário está aprendendo IA.",
]

mem_emb = embedder.encode(memories)
memory_index = faiss.IndexFlatL2(dim)
memory_index.add(mem_emb)

def recall_memory(query, k=2):
    q_emb = embedder.encode([query])
    distances, indices = memory_index.search(q_emb, k)
    return [memories[i] for i in indices[0]]

def add_memory(text):
    memories.append(text)
    new_emb = embedder.encode([text])
    memory_index.add(new_emb)
    return "Memória adicionada."

---
# 8.5 Carregando o LLM (FLAN‑T5)

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 8.6 Módulo de decisão do LLM

O LLM decide:
- consultar banco?
- usar RAG?
- usar memória?
- responder diretamente?

def decide(query):
    prompt = f"""
Você é um assistente inteligente.
Se a pergunta for sobre clientes, responda com: "CONSULTAR: <nome>"
Se for sobre preferências do usuário, responda com: "MEMORIA"
Caso contrário, responda com: "RAG"

Pergunta: {query}
Resposta:
"""
    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=20)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 8.7 Pipeline completo do assistente

def assistant(query):
    decision = decide(query)

    # 1) Consulta ao banco
    if decision.startswith("CONSULTAR:"):
        nome = decision.replace("CONSULTAR:", "").strip()
        return consultar_cliente(nome)

    # 2) Perguntas sobre o usuário → memória
    if decision == "MEMORIA":
        mem = recall_memory(query)
        return "Informações que lembro sobre você:\n" + "\n".join(mem)

    # 3) Caso geral → RAG + memória
    docs_retrieved = retrieve_docs(query)
    mem_retrieved = recall_memory(query)

    context = "\n".join(docs_retrieved + mem_retrieved)

    prompt = f"""
Use o contexto abaixo para responder a pergunta.

Contexto:
{context}

Pergunta:
{query}

Resposta:
"""
    tokens = tokenizer(prompt, return_tensors="tf", truncation=True)
    output = llm.generate(**tokens, max_length=150)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 8.8 Testes do assistente completo

assistant("Qual é o saldo da Carla?")

assistant("O que você sabe sobre meus estudos?")

assistant("Explique o que são redes neurais.")

assistant("Gatos são legais, né?")

---
# 8.9 Adicionando memória dinâmica

add_memory("O usuário gosta de estudar à noite.")

assistant("Qual é meu horário preferido de estudo?")

---
# 8.10 O que você acabou de construir?

✔️ Um chatbot com RAG  
✔️ Um sistema de memória de longo prazo  
✔️ Um banco de dados integrado  
✔️ Um módulo de decisão do LLM  
✔️ Uma arquitetura híbrida completa  

Este é o padrão usado em:
- empresas  
- assistentes internos  
- chatbots corporativos  
- sistemas de suporte técnico  
- agentes autônomos  

---
# 8.11 Conclusão do Módulo 16

✔️ Você domina RAG  
✔️ Você domina memória vetorial  
✔️ Você domina FAISS  
✔️ Você domina arquiteturas híbridas  
✔️ Você construiu um sistema completo  

Parabéns — você concluiu o **Módulo 16**.

O próximo passo é o **Módulo 17 — Agentes Autônomos e Ferramentas (Tool Use)**.